In [6]:
import cv2
import numpy as np

# ===============================
# CONFIGURACIÓN DE LA CÁMARA IP
# ===============================
# Se establece la URL del stream de la cámara IP (Se utilizó la app IP Webcam)
# Asegurarse de que la dirección y el puerto coincidan con los configurados en la aplicación.
url = "http://172.16.238.214:8080/video"
cap = cv2.VideoCapture(url)

# Verifica si la cámara se conectó correctamente
if not cap.isOpened():
    print("No se pudo conectar a la cámara.")
    exit()

print("Cámara conectada. Presiona 'q' para salir.")

# ===============================
# BUCLE PRINCIPAL
# ===============================
# El bucle captura continuamente los fotogramas del video en tiempo real.
while True:
    ret, frame = cap.read()
    if not ret:
        print("Error al recibir frame.")
        break

    img = frame.copy()

    # ============================================
    # BLOQUE 1: PREPROCESAMIENTO
    # ============================================
    # Este bloque convierte la imagen a escala de grises, reduce el ruido
    # y calcula los gradientes para resaltar bordes y contornos del blister.

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.medianBlur(gray, 5)

    # Cálculo del gradiente con Sobel para resaltar bordes horizontales y verticales
    gx = cv2.Sobel(blur, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(blur, cv2.CV_32F, 0, 1, ksize=3)
    grad = cv2.magnitude(gx, gy)
    grad = cv2.normalize(grad, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')

    # Binarización automática mediante método de Otsu
    _, th = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Operación morfológica para cerrar huecos y unir contornos
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
    closed = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel)

    # Detección de contornos externos
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        cv2.imshow("Detección de Huecos y Pastillas", img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    # Se selecciona el contorno de mayor área (blister completo)
    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)

    # Creación de máscara del blister detectado
    mask = np.zeros_like(gray)
    cv2.drawContours(mask, [c], -1, 255, -1)

    blister_roi = cv2.bitwise_and(img, img, mask=mask)

    # Recorte del blister al rectángulo delimitador
    contours_mask, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c_max = max(contours_mask, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c_max)
    offset_x = x
    offset_y = y
    blister_cropped = blister_roi[y:y+h, x:x+w]
    mask_cropped = mask[y:y+h, x:x+w]
    img_cropped = img[y:y+h, x:x+w]

    if blister_cropped.size == 0:
        continue

    # ============================================
    # BLOQUE 2: DETECCIÓN DE PASTILLAS PRESENTES
    # ============================================
    # Este bloque identifica las pastillas presentes mediante segmentación
    # por color en el espacio HSV (rango amarillo y naranja).

    image = blister_cropped.copy()
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Definición de rangos de color
    lower_yellow = np.array([15, 80, 80])
    upper_yellow = np.array([35, 255, 255])
    lower_orange = np.array([5, 80, 80])
    upper_orange = np.array([15, 255, 255])

    # Máscaras para ambos tonos
    mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)
    mask_orange = cv2.inRange(hsv, lower_orange, upper_orange)
    mask_pills = cv2.bitwise_or(mask_yellow, mask_orange)

    # Limpieza morfológica de la máscara
    kernel = np.ones((9, 9), np.uint8)
    mask_cleaned = cv2.morphologyEx(mask_pills, cv2.MORPH_CLOSE, kernel)
    mask_cleaned = cv2.morphologyEx(mask_cleaned, cv2.MORPH_OPEN, kernel)

    # Detección de contornos correspondientes a pastillas
    contours_present, _ = cv2.findContours(mask_cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    present_boxes = []
    areas_pills = []
    for c in contours_present:
        area = cv2.contourArea(c)
        if area < 100:
            continue
        x, y, w, h = cv2.boundingRect(c)
        present_boxes.append((x, y, w, h))
        areas_pills.append(area)

    # Se obtiene el área promedio (mediana) de las pastillas detectadas
    area_prom = np.median(areas_pills) if len(areas_pills) > 0 else 500

    # ============================================
    # BLOQUE 3: DETECCIÓN DE HUECOS
    # ============================================
    # En este bloque se detectan los huecos (espacios vacíos) dentro del blister,
    # generalmente correspondientes a pastillas faltantes.

    lower_white = np.array([0, 0, 100])
    upper_white = np.array([180, 120, 250])
    mask_white = cv2.inRange(hsv, lower_white, upper_white)
    mask_white = cv2.bitwise_and(mask_white, mask_cropped)

    # Limpieza morfológica para mejorar la segmentación de huecos
    kernel = np.ones((9, 9), np.uint8)
    mask_white = cv2.morphologyEx(mask_white, cv2.MORPH_OPEN, kernel, iterations=2)
    mask_white = cv2.morphologyEx(mask_white, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Detección de contornos de huecos
    contours_holes, _ = cv2.findContours(mask_white, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    missing_boxes = []
    centroides_huecos = []  # Lista para almacenar coordenadas de los centroides de huecos

    for c in contours_holes:
        area = cv2.contourArea(c)
        if area < 0.4 * area_prom or area > 1.6 * area_prom:
            continue

        x, y, w, h = cv2.boundingRect(c)
        ratio = w / float(h)
        shape = "hueco" if 0.3 < ratio < 1.2 else "capsula"

        # Cálculo del centroide del contorno
        M = cv2.moments(c)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            cx_global = cx + offset_x
            cy_global = cy + offset_y
            centroides_huecos.append((cx, cy))
            centroides_huecos.append((cx, cy))
        else:
            cx, cy = 0, 0

        missing_boxes.append((x, y, w, h, shape))

    # Impresión de coordenadas de los centroides en la consola
    if len(centroides_huecos) > 0:
        print("Coordenadas de centroides de huecos:", centroides_huecos)

    # ============================================
    # BLOQUE 4: VISUALIZACIÓN EN TIEMPO REAL
    # ============================================
    # Se muestran los resultados de la detección en tiempo real:
    # - Cuadros verdes: pastillas presentes.
    # - Cuadros rojos: huecos o pastillas faltantes.
    # - Círculos azules: centroides de los huecos detectados.

    output = image.copy()

    # Dibujo de pastillas detectadas (verde)
    for (x, y, w, h) in present_boxes:
        cv2.rectangle(output, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Dibujo de huecos (rojo)
    for (x, y, w, h, shape) in missing_boxes:
        cv2.rectangle(output, (x, y), (x + w, y + h), (0, 0, 255), 2)
        cv2.putText(output, shape, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (0, 0, 255), 2)

    # Dibujo de centroides (azul)
    for (cx, cy) in centroides_huecos:
        cv2.circle(output, (cx, cy), 4, (255, 0, 0), -1)
        cv2.putText(output, f"({cx},{cy})", (cx + 5, cy - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

    # Mostrar ventana de salida en tiempo real
    cv2.imshow("Deteccion de Huecos y Pastillas (Tiempo Real)", output)

    # Condición de salida con la tecla 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Liberar recursos al finalizar
cap.release()
cv2.destroyAllWindows()


Cámara conectada. Presiona 'q' para salir.
Coordenadas de centroides de huecos: [(401, 853), (401, 853), (501, 84), (501, 84)]
Coordenadas de centroides de huecos: [(396, 858), (396, 858), (519, 90), (519, 90)]
Coordenadas de centroides de huecos: [(412, 871), (412, 871), (519, 100), (519, 100)]
Coordenadas de centroides de huecos: [(378, 848), (378, 848), (507, 80), (507, 80)]
Coordenadas de centroides de huecos: [(337, 859), (337, 859), (517, 88), (517, 88)]
Coordenadas de centroides de huecos: [(421, 837), (421, 837), (511, 67), (511, 67)]
Coordenadas de centroides de huecos: [(388, 903), (388, 903), (519, 125), (519, 125)]
Coordenadas de centroides de huecos: [(410, 929), (410, 929), (513, 161), (513, 161)]
Coordenadas de centroides de huecos: [(392, 831), (392, 831)]
Coordenadas de centroides de huecos: [(390, 831), (390, 831)]
Coordenadas de centroides de huecos: [(390, 839), (390, 839)]
Coordenadas de centroides de huecos: [(397, 825), (397, 825)]
Coordenadas de centroides de hu